# Station continuity breaks data preparation

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"
DATA_FILE = DATA_DIR / "final_trip_data.parquet"
rides_data = pd.read_parquet(DATA_FILE)

In [3]:
rides_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
4872313,2.0,CLASSIC,2024-08-01 08:26:00,51.500744,-0.202759,1115.0,"Ilchester Place, Kensington",11.269322,2024-08-01 08:12:00,Thursday,51.476885,-0.215896,200029,"Finlay Street, Fulham",2.804527,14m 55s,14.931833,141622479
4863098,2.0,CLASSIC,2024-08-01 13:43:00,51.494224,-0.236770,300037.0,"Ravenscourt Park Station, Hammersmith",8.507715,2024-08-01 13:26:00,Thursday,51.500744,-0.202759,1115,"Ilchester Place, Kensington",2.463470,17m 22s,17.373433,141631864
4809152,2.0,CLASSIC,2024-08-03 12:37:00,51.494499,-0.228188,200166.0,"Southerton Road, Hammersmith",11.829568,2024-08-03 12:34:00,Saturday,51.494224,-0.236770,300037,"Ravenscourt Park Station, Hammersmith",0.594909,3m 1s,3.017400,141687538
4757263,2.0,CLASSIC,2024-08-05 08:58:00,51.514767,-0.225787,200136.0,"BBC White City, White City",9.351433,2024-08-05 08:43:00,Monday,51.494499,-0.228188,200166,"Southerton Road, Hammersmith",2.259820,14m 29s,14.499300,141741326
3204525,2.0,CLASSIC,2024-09-05 12:00:00,51.514441,-0.087587,1201.0,"Bank of England Museum, Bank",11.196041,2024-09-05 11:51:00,Thursday,51.529537,-0.083353,3427,"Fanshaw Street, Hoxton",1.703944,9m 7s,9.131500,142640332


In [4]:
all_bikes_id = rides_data["bike_id"].unique()

In [5]:
len(all_bikes_id)

15739

In [6]:
# ---------- Previous trip information for the same bike ----------
rides_data["previous_end_station_id"] = (
    rides_data
    .groupby("bike_id")["end_station_id"]
    .shift(1)
)

rides_data["previous_end_date"] = (
    rides_data
    .groupby("bike_id")["end_date"]
    .shift(1)
)
rides_data["previous_end_station_name"] = (
    rides_data
    .groupby("bike_id")["end_station_name"]
    .shift(1)
)

In [7]:
rides_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,...,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id,previous_end_station_id,previous_end_date,previous_end_station_name
4872313,2.0,CLASSIC,2024-08-01 08:26:00,51.500744,-0.202759,1115.0,"Ilchester Place, Kensington",11.269322,2024-08-01 08:12:00,Thursday,...,-0.215896,200029,"Finlay Street, Fulham",2.804527,14m 55s,14.931833,141622479,NaN,NaT,NaN
4863098,2.0,CLASSIC,2024-08-01 13:43:00,51.494224,-0.236770,300037.0,"Ravenscourt Park Station, Hammersmith",8.507715,2024-08-01 13:26:00,Thursday,...,-0.202759,1115,"Ilchester Place, Kensington",2.463470,17m 22s,17.373433,141631864,1115.0,2024-08-01 08:26:00,"Ilchester Place, Kensington"
4809152,2.0,CLASSIC,2024-08-03 12:37:00,51.494499,-0.228188,200166.0,"Southerton Road, Hammersmith",11.829568,2024-08-03 12:34:00,Saturday,...,-0.236770,300037,"Ravenscourt Park Station, Hammersmith",0.594909,3m 1s,3.017400,141687538,300037.0,2024-08-01 13:43:00,"Ravenscourt Park Station, Hammersmith"
4757263,2.0,CLASSIC,2024-08-05 08:58:00,51.514767,-0.225787,200136.0,"BBC White City, White City",9.351433,2024-08-05 08:43:00,Monday,...,-0.228188,200166,"Southerton Road, Hammersmith",2.259820,14m 29s,14.499300,141741326,200166.0,2024-08-03 12:37:00,"Southerton Road, Hammersmith"
3204525,2.0,CLASSIC,2024-09-05 12:00:00,51.514441,-0.087587,1201.0,"Bank of England Museum, Bank",11.196041,2024-09-05 11:51:00,Thursday,...,-0.083353,3427,"Fanshaw Street, Hoxton",1.703944,9m 7s,9.131500,142640332,200136.0,2024-08-05 08:58:00,"BBC White City, White City"


# ---------- Detecting discontinuities ----------


In [8]:

discontinuity_mask = (
    rides_data["previous_end_station_id"].notna()
    & (rides_data["previous_end_station_id"] != rides_data["start_station_id"])
)

bike_discontinuities = rides_data[discontinuity_mask].copy()

In [9]:
bike_discontinuities["time_gap"] = (
    bike_discontinuities["start_date"] - bike_discontinuities["previous_end_date"]
)

In [10]:
bike_discontinuities["time_gap_minutes"] = (
    bike_discontinuities["time_gap"].dt.total_seconds() / 60
)

In [11]:
bike_discontinuities_table = bike_discontinuities[
    [
        "bike_id",
        "bike_model",
        "previous_end_date",
        "previous_end_station_id",
        "previous_end_station_name",
        "start_date",
        "start_station_id",
        "start_station_name",
        "end_date",
        "end_station_id",
        "end_station_name",
        "time_gap",
        "time_gap_minutes",
        "trip_id"
    ]
]

In [12]:
bike_discontinuities_table.head(20)

,bike_id,bike_model,previous_end_date,previous_end_station_id,previous_end_station_name,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,time_gap,time_gap_minutes,trip_id
3204525,2.0,CLASSIC,2024-08-05 08:58:00,200136.0,"BBC White City, White City",2024-09-05 11:51:00,3427,"Fanshaw Street, Hoxton",2024-09-05 12:00:00,1201.0,"Bank of England Museum, Bank",31 days 02:53:00,44813.0,142640332
4691497,2.0,CLASSIC,2024-09-06 22:43:00,1223.0,"Buxton Street 1, Shoreditch",2024-08-07 12:34:00,300026,"All Saints' Road, Portobello",2024-08-07 12:59:00,1231.0,"Queen's Gate, Kensington Gardens",-31 days +13:51:00,-43809.0,141808987
3149862,2.0,CLASSIC,2024-08-07 15:01:00,1140.0,"Grosvenor Road, Pimlico",2024-09-07 16:15:00,1223,"Buxton Street 1, Shoreditch",2024-09-07 16:28:00,300049.0,"Aylward Street, Stepney",31 days 01:14:00,44714.0,142696481
4667203,2.0,CLASSIC,2024-09-07 20:13:00,3494.0,"Foley Street, Fitzrovia",2024-08-08 08:40:00,1140,"Grosvenor Road, Pimlico",2024-08-08 08:45:00,2683.0,"Millbank Tower, Pimlico",-31 days +12:27:00,-43893.0,141834075
3135144,2.0,CLASSIC,2024-08-08 19:28:00,300045.0,"Albert Bridge Road, Battersea Park",2024-09-08 11:34:00,3494,"Foley Street, Fitzrovia",2024-09-08 11:50:00,3504.0,"Moor Street, Soho",30 days 16:06:00,44166.0,142711580
4640784,2.0,CLASSIC,2024-09-08 17:47:00,10625.0,"Penfold Street, Marylebone",2024-08-09 08:31:00,300045,"Albert Bridge Road, Battersea Park",2024-08-09 08:36:00,200183.0,"Battersea Park Road, Nine Elms",-31 days +14:44:00,-43756.0,141861195
3110078,2.0,CLASSIC,2024-08-09 09:33:00,200201.0,"Battersea Church Road, Battersea",2024-09-09 10:39:00,10625,"Penfold Street, Marylebone",2024-09-09 10:52:00,22179.0,"Exhibition Road, Knightsbridge",31 days 01:06:00,44706.0,142737254
3099853,2.0,CLASSIC,2024-09-09 10:52:00,22179.0,"Exhibition Road, Knightsbridge",2024-09-09 17:49:00,300043,"St. Peter's Terrace, Fulham",2024-09-09 18:23:00,200059.0,"Crisp Road, Hammersmith",0 days 06:57:00,417.0,142747787
3026271,2.0,CLASSIC,2024-08-12 22:10:00,1059.0,"Albert Embankment, Vauxhall",2024-09-12 11:01:00,300087,"Imperial Road, Sands End",2024-09-12 11:15:00,300033.0,"Putney Rail Station, Putney",30 days 12:51:00,43971.0,142823354
2929523,2.0,CLASSIC,2024-09-14 23:20:00,300006.0,"Clapham South, Clapham South",2024-09-16 07:19:00,300051,"Hibbert Street, Battersea",2024-09-16 07:42:00,1190.0,"Kennington Lane Rail Bridge, Vauxhall",1 days 07:59:00,1919.0,142922881


In [13]:
print(f"Number of discontinuities: {len(bike_discontinuities_table)}")

Number of discontinuities: 1334475


In [14]:
discontinuity_percent = (
    len(bike_discontinuities_table) / len(rides_data) * 100
)

print(f"Percentage of discontinuous trips: {discontinuity_percent:.3f}%")

Percentage of discontinuous trips: 7.658%


In [15]:
rides_data_continuous = rides_data[~discontinuity_mask].copy()

In [16]:
rides_data_continuous.to_parquet(DATA_DIR /
    "rides_data_without_discontinuities.parquet",
    index=False
)